In [2]:
import os, time, subprocess, pythoncom, psutil
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS & CONFIG
# ══════════════════════════════════════════════════════════════════════════════
first_glob       = os.path.expanduser("~").replace("\\", "/")
PARQUET_PATH     = (f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
                    f"/BI_Task/CODE/Resources/excalibur_raw.parquet")
IC_DETAILS_PATH  = (f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
                    f"/BI_Task/CODE/Resources/IC_HCM_Details_Log.parquet")

DISPLAY_NOTEBOOK = True
SEND_EMAIL       = False

EMAIL_TO = "huuchinh.nguyen@concentrix.com"
EMAIL_CC = "huuchinh.nguyen@concentrix.com"

OVERRIDE_DATE = None   # None = auto D-1 | "2026-05-28"

if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"⚠️  OVERRIDE: {OVERRIDE_DATE}")
else:
    report_date = datetime.now() - timedelta(days=1)
    print(f"✓ AUTO D-1 = {report_date.strftime('%Y-%m-%d')}")

report_date_d = report_date.date()
report_date_s = report_date.strftime("%d-%b-%Y")
report_now    = datetime.now()
EMAIL_SUBJECT = f"Expedia VN — IC & Staffing Attainment Report as of {report_date_s}"

SITES   = ["VN", "KOL", "PUN", "CAI"]
LOBS    = ["Lodging chat", "Non-Lodging chat"]
LOB_MAP = {"Lodging chat": "LG Chat", "Non-Lodging chat": "NL Chat"}

SITE_MAP = {
    "Concentrix (Ho Chi Minh City)": "VN",
    "Concentrix (Kolkata)":          "KOL",
    "Concentrix (Pune)":             "PUN",
    "Concentrix (Cairo)":            "CAI",
}

IC_TARGET    = 90.0
SA_TARGET    = 85.0
IC_FAIL_RED  = 4

DAILY_FROM       = "2026-06-08" #"2026-05-28"
DAILY_TO         = None
N_DAILY_AUTO     = 6
N_WEEKLY_WEEKS   = 8
N_MONTHLY_MONTHS = 6

DAILY_METRICS   = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]
WEEKLY_METRICS  = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]
MONTHLY_METRICS = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD PARQUET — excalibur_raw
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading excalibur_raw.parquet...")
df_pl = pl.read_parquet(PARQUET_PATH)
df    = df_pl.to_pandas()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()

if "Site_Abbr" not in df.columns:
    df["Site_Abbr"] = df["Site"].map(SITE_MAP).fillna(df["Site"])

df["IC_Pass"] = (df["Interval Compliance (Pct)"] == 1).astype(int)
df["IC_Fail"] = (df["Interval Compliance (Pct)"] == 0).astype(int)
print(f"✓ excalibur: {len(df):,} | {df['Date'].min().date()} → {df['Date'].max().date()}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD PARQUET — IC HCM Details Log
# ══════════════════════════════════════════════════════════════════════════════
print("📂 Loading IC_HCM_Details_Log.parquet...")
df_ic = pd.read_parquet(IC_DETAILS_PATH)
df_ic["PST_Date"] = pd.to_datetime(df_ic["PST_Date"], errors="coerce").dt.normalize()
df_ic["VNT_Date"] = pd.to_datetime(df_ic["VNT_Date"], errors="coerce").dt.normalize()
df_ic["LOB_disp"] = df_ic["LOB"].map(LOB_MAP).fillna(df_ic["LOB"])
print(f"✓ IC Details: {len(df_ic):,} | {df_ic['PST_Date'].min().date()} → {df_ic['PST_Date'].max().date()}")

# ── Auto daily range — lấy theo PST Date max trong excalibur ─────────────────
if DAILY_TO is None:
    _excalibur_max = df["Date"].max().date()
    daily_to       = _excalibur_max - timedelta(days=1)
    print(f"  Excalibur max PST Date: {_excalibur_max} → daily_to: {daily_to}")
else:
    daily_to = datetime.strptime(DAILY_TO, "%Y-%m-%d").date()

if DAILY_FROM is None:
    _all_dates = sorted(df[df["Date"].dt.date <= daily_to]["Date"].dt.date.unique())
    daily_from = _all_dates[-N_DAILY_AUTO] if len(_all_dates) >= N_DAILY_AUTO else _all_dates[0]
else:
    daily_from = datetime.strptime(DAILY_FROM, "%Y-%m-%d").date()

print(f"✓ Daily range: {daily_from} → {daily_to}")
print(f"✓ Subject    : {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE IC METRICS — dedup per interval
# ══════════════════════════════════════════════════════════════════════════════
def compute_ic(df_sub: pd.DataFrame) -> dict:
    ic_dedup = (
        df_sub
        .groupby(["Date", "PST_Interval"], as_index=False)
        .agg(ic_val=("Interval Compliance (Pct)", "first"))
        .dropna(subset=["ic_val"])
    )
    total   = len(ic_dedup)
    ic_pass = int((ic_dedup["ic_val"] == 1).sum())
    ic_fail = int((ic_dedup["ic_val"] == 0).sum())
    ic_pct  = round(ic_pass / total * 100, 2) if total > 0 else None

    grp = (
        df_sub
        .groupby(["Date", "PST_Interval"], as_index=False)
        .agg(
            tot_prod=("Total Productive Hours", "max"),
            tot_psp =("Total PSP",              "max"),
        )
    )
    grp    = grp[grp["tot_psp"] > 0]
    sa_pct = (
        round((grp["tot_prod"] / grp["tot_psp"]).mean() * 100, 2)
        if len(grp) > 0 else None
    )
    return {
        "IC Pass": ic_pass,
        "IC Fail": ic_fail,
        "IC %":    ic_pct,
        "Staffing Attainment %": sa_pct,
    }

# ══════════════════════════════════════════════════════════════════════════════
# PIVOT BUILDER
# ══════════════════════════════════════════════════════════════════════════════
def build_pivot(df_in: pd.DataFrame, time_col: str, metrics: list,
                ordered_periods: list = None):
    periods  = ordered_periods if ordered_periods else sorted(df_in[time_col].dropna().unique())
    lob_keys = list(LOB_MAP.values()) + ["Total"]
    records  = {(lob, m): {} for lob in lob_keys for m in metrics}

    for p in periods:
        df_p = df_in[df_in[time_col] == p]
        for lob_raw, lob_key in LOB_MAP.items():
            df_lob = df_p[df_p["LOB"] == lob_raw]
            if df_lob.empty: continue
            m = compute_ic(df_lob)
            for mk in metrics:
                records[(lob_key, mk)][p] = m.get(mk)
        m_all = compute_ic(df_p[df_p["LOB"].isin(LOBS)])
        for mk in metrics:
            records[("Total", mk)][p] = m_all.get(mk)

    rows = []
    for lob in lob_keys:
        for mk in metrics:
            row = {"LOB": lob, "Metric": mk}
            for p in periods:
                row[p] = records.get((lob, mk), {}).get(p)
            rows.append(row)

    return pd.DataFrame(rows), periods

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL PIVOTS
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ Building Daily pivot...")
df_daily = df[
    (df["Date"].dt.date >= daily_from) &
    (df["Date"].dt.date <= daily_to)
].copy()
df_daily["_D"] = df_daily["Date"].dt.strftime("%Y-%m-%d")
daily_piv, daily_periods = build_pivot(
    df_daily.rename(columns={"_D": "Date_Label"}),
    "Date_Label", DAILY_METRICS
)
print(f"✓ Daily: {daily_periods}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD FAILED INTERVALS — excalibur_raw (Section 2)
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ Building Excalibur failed intervals...")
df_fail_ex = df_daily[df_daily["Interval Compliance (Pct)"] == 0].copy()

grp_base = (
    df_fail_ex
    .groupby(["Date","LOB","PST_Interval_Range","PST_Interval"], as_index=False)
    .agg(
        Total_PSP              =("Total PSP",              "max"),
        Total_Productive_Hours =("Total Productive Hours", "max"),
    )
)

site_dfs = []
for site in SITES:
    s = (
        df_fail_ex[df_fail_ex["Site_Abbr"] == site]
        .groupby(["Date","LOB","PST_Interval"], as_index=False)
        .agg(prod=("Productive Hours","sum"), msp_site=("MSP site wise","sum"))
    )
    s[site] = s["prod"] - s["msp_site"]
    site_dfs.append(s[["Date","LOB","PST_Interval",site]])

fail_site = grp_base.copy()
for s_df in site_dfs:
    fail_site = fail_site.merge(s_df, on=["Date","LOB","PST_Interval"], how="left")

vnt_map = (
    df_fail_ex.groupby(["Date","LOB","PST_Interval"], as_index=False)["VNT_Datetime"].first()
)
vnt_map["VNT_Interval"] = pd.to_datetime(
    vnt_map["VNT_Datetime"], errors="coerce"
).dt.strftime("%H:%M")

fail_site = fail_site.merge(
    vnt_map[["Date","LOB","PST_Interval","VNT_Interval"]],
    on=["Date","LOB","PST_Interval"], how="left"
)
fail_site = fail_site.rename(columns={
    "Total_PSP":              "Total PSP",
    "Total_Productive_Hours": "Total Productive Hours",
})
fail_site["SA %"] = np.where(
    fail_site["Total PSP"] > 0,
    (fail_site["Total Productive Hours"] / fail_site["Total PSP"] * 100).round(2),
    np.nan
)
fail_site["LOB_disp"] = fail_site["LOB"].map(LOB_MAP).fillna(fail_site["LOB"])
fail_site["Date_str"] = fail_site["Date"].dt.strftime("%Y-%m-%d")
fail_site = fail_site.sort_values(["Date","LOB","PST_Interval"]).reset_index(drop=True)
print(f"✓ Excalibur failed: {len(fail_site)}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD FAILED INTERVALS — IC HCM Details (Section 3)
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ Building IC Details failed intervals...")
df_ic_fail = df_ic[
    (df_ic["IC Status"] == "Missed") &
    (df_ic["PST_Date"].dt.date >= daily_from) &
    (df_ic["PST_Date"].dt.date <= daily_to)
].copy()
df_ic_fail["Date_str"]     = df_ic_fail["PST_Date"].dt.strftime("%Y-%m-%d")
df_ic_fail["VNT_Date_str"] = df_ic_fail["VNT_Date"].dt.strftime("%Y-%m-%d")
df_ic_fail["SA_pct"]       = (df_ic_fail["Staffing Attainment (Pct)"] * 100).round(1)
df_ic_fail = df_ic_fail.sort_values(["PST_Date","LOB","PST_Intervals"]).reset_index(drop=True)
print(f"✓ IC Details failed: {len(df_ic_fail)}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD WEEKLY & MONTHLY PIVOTS
# ══════════════════════════════════════════════════════════════════════════════
print("⏳ Building Weekly pivot...")
df["_Week_Period"] = df["Date"].dt.to_period("W")
df["Week_Label"]   = df["_Week_Period"].apply(
    lambda p: f"{p.start_time.strftime('%m/%d')}~{p.end_time.strftime('%m/%d')}"
    if pd.notna(p) else None
)
week_map = (
    df[df["Date"].dt.date <= daily_to]
    .groupby("_Week_Period", as_index=False)["Week_Label"].first()
    .sort_values("_Week_Period")
)
week_map = week_map[
    week_map["_Week_Period"].apply(lambda p: p.start_time.year) >= daily_to.year
]
recent_week_labels = week_map["Week_Label"].tolist()[-N_WEEKLY_WEEKS:]
df_weekly = df[df["Week_Label"].isin(recent_week_labels)].copy()
weekly_piv, weekly_periods = build_pivot(
    df_weekly.rename(columns={"Week_Label": "Week"}),
    "Week", WEEKLY_METRICS, ordered_periods=recent_week_labels
)
print(f"✓ Weekly ({len(weekly_periods)}): {weekly_periods}")

print("⏳ Building Monthly pivot...")
df["Month_Label"] = df["Date"].dt.strftime("%Y-%m")
all_months    = sorted(df[df["Date"].dt.date <= daily_to]["Month_Label"].dropna().unique())
recent_months = all_months[-N_MONTHLY_MONTHS:]
df_monthly    = df[df["Month_Label"].isin(recent_months)].copy()
monthly_piv, monthly_periods = build_pivot(
    df_monthly.rename(columns={"Month_Label": "Month"}),
    "Month", MONTHLY_METRICS, ordered_periods=recent_months
)
print(f"✓ Monthly ({len(monthly_periods)}): {monthly_periods}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG   = "#d4f4e2"; MET_FG  = "#1a5c2a"
WARN_BG  = "#fff3cd"; WARN_FG = "#7a5200"
MISS_BG  = "#fde8ea"; MISS_FG = "#9b1c2a"
HDR_DARK = "#1a3a5c"; HDR_MID = "#1f5c99"
BANNER_C = "#8b0020"

LG_HDR = "#1a6b3a"; LG_ROW = "#f0faf4"; LG_SEP = "#d4edda"
NL_HDR = "#0e3d7a"; NL_ROW = "#f0f5ff"; NL_SEP = "#cce0ff"
TOT_HDR = "#4a5568"; TOT_ROW = "#edf2f7"; TOT_SEP = "#cbd5e0"; TOT_FG = "#2d3748"

SEC_BADGE_BG = "#e6a817"; SEC_BADGE_FG = "#1a1a1a"
FONT = "font-family:Arial,sans-serif;font-size:11px;"
TH_S = (f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
        f"white-space:nowrap;text-align:center;"
        f"border:1px solid rgba(255,255,255,0.2);")
TD_S = f"{FONT}padding:4px 8px;border:1px solid #e2e8f0;white-space:nowrap;"

SITE_COLORS = {
    "VN":  {"bar":"#2e86c1","hdr":"#2471a3"},
    "KOL": {"bar":"#1e8449","hdr":"#1a7a42"},
    "PUN": {"bar":"#b7950b","hdr":"#9a7d0a"},
    "CAI": {"bar":"#6c3483","hdr":"#5b2c6f"},
}

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #e2e8f0;text-align:right;background:#fff}}
.t tbody td.lbl{{text-align:left}}
.sep-lg td{{background:{LG_SEP};color:{LG_HDR};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {LG_HDR}}}
.sep-nl td{{background:{NL_SEP};color:{NL_HDR};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {NL_HDR}}}
.sep-tot td{{background:{TOT_SEP};color:{TOT_FG};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {TOT_HDR}}}
.r-tot td{{background:{TOT_ROW}!important;color:{TOT_FG}!important;font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.warn{{background:{WARN_BG}!important;color:{WARN_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.fail-red{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.fail-grn{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;
   background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};
   padding:4px 12px;margin:24px 0 4px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;
   border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;
   margin:0 0 10px;border-radius:0 3px 3px 0}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _th(l, bg=HDR_MID):
    return f'<th style="{TH_S}background:{bg};">{l}</th>'

def _thl(l, bg=HDR_DARK):
    return f'<th style="{TH_S}background:{bg};text-align:left;">{l}</th>'

def _sec(num, title, note, em=False):
    bs = (f"display:inline-block;font-size:12px;font-weight:bold;"
          f"background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};"
          f"padding:4px 12px;margin:24px 0 4px;border-radius:3px;")
    ns = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
          f"border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;"
          f"margin:0 0 10px;border-radius:0 3px 3px 0;display:block;")
    if em:
        return (f'<p style="margin:24px 0 4px;">'
                f'<span style="{bs}">{num}. {title}</span></p>'
                f'<p style="{ns}">{note}</p>')
    return (f'<div style="margin:24px 0 4px;">'
            f'<span class="sec-badge">{num}. {title}</span></div>'
            f'<div class="note">{note}</div>')

def spacer(em=False):
    if em:
        return ('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                '<tr><td style="height:24px;font-size:1px;">&nbsp;</td></tr></table>')
    return '<div style="height:24px;"></div>'

def lgd(inline=False):
    sp = "padding:2px 8px;margin-right:8px;font-weight:bold;"
    fs = f"{FONT}font-size:11px;"
    if inline:
        return (f'<p style="{fs}margin:6px 0 16px 0;">'
                f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met(&ge;{IC_TARGET:.0f}%)</span>'
                f'<span style="{sp}background:{WARN_BG};color:{WARN_FG}">&#9632; Near</span>'
                f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></p>')
    return (f'<div style="{fs}padding:8px 0 12px;">'
            f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met(&ge;{IC_TARGET:.0f}%)</span>'
            f'<span style="{sp}background:{WARN_BG};color:{WARN_FG}">&#9632; Near</span>'
            f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></div>')

# ══════════════════════════════════════════════════════════════════════════════
# COLORING HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def color_ic(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    fv = float(v)
    if fv >= IC_TARGET:         cls, bg, fg = "met",  MET_BG,  MET_FG
    elif fv >= IC_TARGET * 0.9: cls, bg, fg = "warn", WARN_BG, WARN_FG
    else:                       cls, bg, fg = "miss", MISS_BG, MISS_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_sa(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    fv = float(v)
    if fv >= SA_TARGET:         cls, bg, fg = "met",  MET_BG,  MET_FG
    elif fv >= SA_TARGET * 0.9: cls, bg, fg = "warn", WARN_BG, WARN_FG
    else:                       cls, bg, fg = "miss", MISS_BG, MISS_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_fail(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    cls = "fail-red" if int(v) > IC_FAIL_RED else "fail-grn"
    bg  = MISS_BG if int(v) > IC_FAIL_RED else MET_BG
    fg  = MISS_FG if int(v) > IC_FAIL_RED else MET_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_sa_detail(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "#fff", "#000"
    fv = float(v)
    if fv >= 95:          return MET_BG,  MET_FG
    elif fv >= 95 * 0.9:  return WARN_BG, WARN_FG
    else:                 return MISS_BG, MISS_FG

# ══════════════════════════════════════════════════════════════════════════════
# FORMAT VALUE
# ══════════════════════════════════════════════════════════════════════════════
PCT_METRICS = {"IC %", "Staffing Attainment %"}
INT_METRICS = {"IC Pass", "IC Fail"}

def fmt_val(metric, v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "&#8212;"
    if metric in PCT_METRICS: return f"{float(v):.1f}%"
    if metric in INT_METRICS: return f"{int(v):,}"
    return str(v)

# ══════════════════════════════════════════════════════════════════════════════
# PIVOT TABLE RENDERER
# ══════════════════════════════════════════════════════════════════════════════
LOB_STYLE = {
    "LG Chat": (LG_ROW, LG_HDR, LG_SEP, "sep-lg"),
    "NL Chat": (NL_ROW, NL_HDR, NL_SEP, "sep-nl"),
    "Total":   (TOT_ROW, TOT_FG, TOT_SEP, "sep-tot"),
}

def render_pivot_table(piv: pd.DataFrame, periods: list,
                       is_daily: bool = False,
                       show_total: bool = True,
                       em: bool = False) -> str:
    metrics  = piv["Metric"].unique().tolist()
    lob_keys = ["LG Chat", "NL Chat"] + (["Total"] if show_total else [])
    tc       = "" if em else 'class="t" '

    h = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    h.append(_thl("LOB"))
    h.append(_thl("Metric"))
    for p in periods:
        h.append(_th(p))
    h.append('</tr></thead><tbody>')

    prev_lob = None
    for lob in lob_keys:
        row_bg, lob_col, sep_bg, sep_cls = LOB_STYLE.get(lob, (TOT_ROW, TOT_FG, TOT_SEP, "sep-tot"))
        is_tot = lob == "Total"

        if lob != prev_lob:
            prev_lob = lob
            span = 2 + len(periods)
            if em:
                h.append(f'<tr><td colspan="{span}" style="{FONT}background:{sep_bg};'
                         f'color:{lob_col};font-weight:bold;padding:4px 12px;'
                         f'border-left:3px solid {lob_col}">{lob}</td></tr>')
            else:
                h.append(f'<tr class="{sep_cls}"><td colspan="{span}">{lob}</td></tr>')

        for mk in metrics:
            sub = piv[(piv["LOB"] == lob) & (piv["Metric"] == mk)]
            if sub.empty: continue

            if mk == "IC %":                    cfn = color_ic
            elif mk == "Staffing Attainment %":  cfn = color_sa
            elif mk == "IC Fail" and is_daily:   cfn = color_fail
            else:                               cfn = None

            tr_cls = "r-tot" if is_tot else ""

            if em:
                h.append('<tr>')
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;'
                         f'color:{lob_col};font-weight:{"bold" if is_tot else "normal"}">{lob}</td>')
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{mk}</td>')
                for p in periods:
                    v   = sub.iloc[0].get(p)
                    val = fmt_val(mk, v)
                    if cfn:
                        inl = cfn(v, em=True)[1]
                        h.append(f'<td style="{TD_S}{inl}text-align:right;">{val}</td>')
                    else:
                        h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')
                h.append('</tr>')
            else:
                h.append(f'<tr class="{tr_cls}">')
                h.append(f'<td class="lbl" style="{TD_S}background:{row_bg};'
                         f'color:{lob_col};font-weight:bold">{lob}</td>')
                h.append(f'<td class="lbl" style="{TD_S}background:{row_bg}">{mk}</td>')
                for p in periods:
                    v   = sub.iloc[0].get(p)
                    val = fmt_val(mk, v)
                    if cfn:
                        cls = cfn(v)[0]
                        h.append(f'<td class="{cls}" style="{TD_S}text-align:right;'
                                 f'background:{row_bg}">{val}</td>')
                    else:
                        h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')
                h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# EXCALIBUR FAILED TABLE RENDERER (Section 2)
# ══════════════════════════════════════════════════════════════════════════════
def render_excalibur_fail_table(df_fail_in: pd.DataFrame, em: bool = False) -> str:
    if df_fail_in.empty:
        return (f'<p style="{FONT}color:{MET_FG};background:{MET_BG};'
                f'padding:8px 12px;border-radius:4px;">✅ No failed intervals.</p>')

    COLS = [
        ("Date_str",               "Date"),
        ("LOB_disp",               "LOB"),
        ("PST_Interval_Range",     "PST Interval"),
        ("VNT_Interval",           "VNT Interval"),
        ("Total PSP",              "Forecast Productive"),
        ("Total Productive Hours", "Total Productive Hours"),
        ("SA %",                   "Staffing Attainment %"),
    ] + [(site, site) for site in SITES]

    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    for col_key, col_hdr in COLS:
        if col_key in SITES:
            h.append(f'<th style="{TH_S}background:{SITE_COLORS[col_key]["hdr"]};">{col_hdr}</th>')
        elif col_key in ("Total PSP","Total Productive Hours","SA %"):
            h.append(_th(col_hdr, bg="#4a5568"))
        else:
            h.append(_thl(col_hdr))
    h.append('</tr></thead><tbody>')

    prev_date = None
    for _, row in df_fail_in.iterrows():
        lob_disp = row.get("LOB_disp","")
        date_str = row.get("Date_str","")
        row_bg   = LG_ROW if lob_disp == "LG Chat" else NL_ROW
        lob_col  = LG_HDR if lob_disp == "LG Chat" else NL_HDR

        if date_str != prev_date:
            prev_date = date_str
            h.append(f'<tr><td colspan="{len(COLS)}" style="{FONT}background:#1E1E2E;'
                     f'color:#E8E8FF;font-weight:bold;padding:5px 12px;">{date_str}</td></tr>')

        h.append('<tr>')
        for col_key, _ in COLS:
            v = row.get(col_key)
            is_null = v is None or (isinstance(v, float) and pd.isna(v))

            if col_key == "Date_str":
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{date_str}</td>')
            elif col_key == "LOB_disp":
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;'
                         f'color:{lob_col};font-weight:bold">{lob_disp}</td>')
            elif col_key == "SA %":
                if is_null:
                    val = "&#8212;"; bg_c, fg_c = row_bg, "#000"
                else:
                    fv_ = float(v); val = f"{fv_:.1f}%"
                    bg_c, fg_c = color_sa_detail(fv_)
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:right;">{val}</td>')
            elif col_key in SITES:
                if is_null:
                    val = "&#8212;"; bg_c, fg_c, fw = row_bg, "#000", "normal"
                else:
                    fv_ = float(v); val = f"{fv_:.2f}"
                    if fv_ > 0:   bg_c, fg_c, fw = MET_BG,  MET_FG,  "bold"
                    elif fv_ < 0: bg_c, fg_c, fw = MISS_BG, MISS_FG, "bold"
                    else:         bg_c, fg_c, fw = WARN_BG, WARN_FG, "bold"
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:{fw};text-align:right;">{val}</td>')
            elif col_key in ("Total PSP","Total Productive Hours"):
                val = f"{float(v):.2f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')
            else:
                val = str(v) if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{val}</td>')
        h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# IC DETAILS TABLE RENDERER (Section 3)
# Logic coloring:
#   Actual Heads vs Scheduled OpenTime: Actual < Scheduled → red, else green
#   Actual Break  vs Scheduled Break:   Actual >= Scheduled → red, else green
#   Actual Lunch  vs Scheduled Lunch:   Actual >= Scheduled → red, else green
#   Actual T/C    vs Scheduled T/C:     Actual >= Scheduled → red, else green
#   Scheduled Leave > 0 → red, else green
#   Scheduled NCNS  > 0 → red, else green
#   Scheduled OpenTime, Site Req Heads → no coloring (plain)
# ══════════════════════════════════════════════════════════════════════════════
IC_DETAIL_COLS = [
    ("LOB_disp",                    "LOB"),
    ("Date_str",                    "PST Date"),
    ("PST_Interval_Range",          "PST Interval"),
    ("VNT_Date_str",                "VNT Date"),
    ("VNT_Interval_Range",          "VNT Interval"),
    ("IC Status",                   "IC Status"),
    ("SA_pct",                      "SA %"),
    ("Site Req Heads",              "Req Heads"),
    ("Scheduled_Open_Time",         "Scheduled OpenTime"),
    ("Site Actual Heads",           "Actual Heads"),
    ("Scheduled_Break",             "Scheduled Break"),
    ("Actual_Break",                "Actual Break"),
    ("Scheduled_Lunch",             "Scheduled Lunch"),
    ("Actual_Lunch",                "Actual Lunch"),
    ("Scheduled_Training/Coaching", "Scheduled Training/Coaching"),
    ("Actual_Training_Coaching",    "Actual Training/Coaching"),
    ("Scheduled_Leave",             "Scheduled Leave (AL+CO)"),
    ("Scheduled_NCNS",              "Scheduled NCNS"),
]


IC_HDR_COLORS = {
    "LOB":                          HDR_DARK,
    "PST Date":                     HDR_DARK,
    "PST Interval":                 HDR_DARK,
    "VNT Date":                     HDR_DARK,
    "VNT Interval":                 HDR_DARK,
    "IC Status":                    "#c0003c",
    "SA %":                         "#5a3e00",
    "Req Heads":                    "#4a5568",
    "Scheduled OpenTime":           "#1a6b3a",
    "Actual Heads":                 "#1a6b3a",
    "Scheduled Break":              "#7a5200",
    "Actual Break":                 "#7a5200",
    "Scheduled Lunch":              "#7a5200",
    "Actual Lunch":                 "#7a5200",
    "Scheduled Training/Coaching":  "#7a5200",
    "Actual Training/Coaching":     "#7a5200",
    "Scheduled Leave (AL+CO)":      "#5b2c6f",
    "Scheduled NCNS":               "#5b2c6f",
}

IC_SUM_COLS = [
    "Site Req Heads","Scheduled_Open_Time","Site Actual Heads",  # ← đổi
    "Scheduled_Break","Actual_Break","Scheduled_Lunch","Actual_Lunch",
    "Scheduled_Training/Coaching","Actual_Training_Coaching",
    "Scheduled_Leave","Scheduled_NCNS",
]

def _ic_cell_color(col_key: str, v, row: pd.Series, row_bg: str):
    is_null = v is None or (isinstance(v, float) and pd.isna(v))
    if is_null:
        return row_bg, "#000", "normal"

    fv = float(v)

    # Site Actual Heads vs Scheduled OpenTime
    if col_key == "Site Actual Heads":
        sch = row.get("Scheduled_Open_Time")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            return (MISS_BG, MISS_FG, "bold") if fv < float(sch) else (MET_BG, MET_FG, "bold")
        return row_bg, "#000", "normal"

    if col_key == "Actual_Break":
        sch = row.get("Scheduled_Break")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            return (MISS_BG, MISS_FG, "bold") if fv >= float(sch) else (MET_BG, MET_FG, "bold")
        return row_bg, "#000", "normal"

    if col_key == "Actual_Lunch":
        sch = row.get("Scheduled_Lunch")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            return (MISS_BG, MISS_FG, "bold") if fv >= float(sch) else (MET_BG, MET_FG, "bold")
        return row_bg, "#000", "normal"

    if col_key == "Actual_Training_Coaching":
        sch = row.get("Scheduled_Training/Coaching")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            return (MISS_BG, MISS_FG, "bold") if fv >= float(sch) else (MET_BG, MET_FG, "bold")
        return row_bg, "#000", "normal"

    if col_key == "Scheduled_Leave":
        return (MISS_BG, MISS_FG, "bold") if fv > 0 else (MET_BG, MET_FG, "normal")

    if col_key == "Scheduled_NCNS":
        return (MISS_BG, MISS_FG, "bold") if fv > 0 else (MET_BG, MET_FG, "normal")

    return row_bg, "#000", "normal"


def render_ic_details_table(df_in: pd.DataFrame, em: bool = False) -> str:
    if df_in.empty:
        return (f'<p style="{FONT}color:{MET_FG};background:{MET_BG};'
                f'padding:8px 12px;border-radius:4px;">✅ No missed intervals.</p>')

    # Cols that get comparison coloring
    COLORED_COLS = {
        "Site Actual Heads",           # ← đổi từ "Actual Heads"
        "Actual_Break", "Actual_Lunch",
        "Actual_Training_Coaching", "Scheduled_Leave", "Scheduled_NCNS"
    }

    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    for col_key, col_hdr in IC_DETAIL_COLS:
        bg = IC_HDR_COLORS.get(col_hdr, HDR_DARK)
        h.append(f'<th style="{TH_S}background:{bg};">{col_hdr}</th>')
    h.append('</tr></thead><tbody>')

    prev_date = None
    prev_lob  = None

    for _, row in df_in.iterrows():
        lob_disp = row.get("LOB_disp","")
        date_str = row.get("Date_str","")
        row_bg   = LG_ROW if lob_disp == "LG Chat" else NL_ROW
        lob_col  = LG_HDR if lob_disp == "LG Chat" else NL_HDR
        span     = len(IC_DETAIL_COLS)

        if date_str != prev_date:
            prev_date = date_str; prev_lob = None
            h.append(f'<tr><td colspan="{span}" style="{FONT}background:#1E1E2E;'
                     f'color:#E8E8FF;font-weight:bold;padding:5px 12px;">{date_str}</td></tr>')

        if lob_disp != prev_lob:
            prev_lob = lob_disp
            sep_bg   = LG_SEP if lob_disp == "LG Chat" else NL_SEP
            h.append(f'<tr><td colspan="{span}" style="{FONT}background:{sep_bg};'
                     f'color:{lob_col};font-weight:bold;padding:3px 12px;'
                     f'border-left:3px solid {lob_col}">{lob_disp}</td></tr>')

        h.append('<tr>')
        for col_key, col_hdr in IC_DETAIL_COLS:
            v       = row.get(col_key)
            is_null = v is None or (isinstance(v, float) and pd.isna(v))

            # ── IC Status ─────────────────────────────────────────────────────
            if col_key == "IC Status":
                ic_val = str(v) if not is_null else "&#8212;"
                bg_c   = MISS_BG if ic_val == "Missed" else MET_BG
                fg_c   = MISS_FG if ic_val == "Missed" else MET_FG
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:center;">{ic_val}</td>')

            # ── SA % ─────────────────────────────────────────────────────────
            elif col_key == "SA_pct":
                if is_null:
                    val = "&#8212;"; bg_c, fg_c = row_bg, "#000"
                else:
                    val = f"{float(v):.1f}%"
                    bg_c, fg_c = color_sa_detail(float(v))
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:right;">{val}</td>')

            # ── Label cols ────────────────────────────────────────────────────
            elif col_key in ("LOB_disp","Date_str","PST_Interval_Range","VNT_Date_str","VNT_Interval_Range"):
                val = str(v) if not is_null else "&#8212;"
                fc  = lob_col if col_key == "LOB_disp" else "#000"
                fw  = "bold"  if col_key == "LOB_disp" else "normal"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;'
                        f'color:{fc};font-weight:{fw};">{val}</td>')

            # ── Comparison-colored cols ───────────────────────────────────────
            elif col_key in COLORED_COLS:
                bg_c, fg_c, fw = _ic_cell_color(col_key, v, row, row_bg)
                val = f"{float(v):.1f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:{fw};text-align:right;">{val}</td>')

            # ── Plain numeric cols (Req Heads, Scheduled cols) ─────────────────
            else:
                val = f"{float(v):.1f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')

        h.append('</tr>')

    # ── Grand Total ───────────────────────────────────────────────────────────
    gt_sa = (df_in["Staffing Attainment (Pct)"].mean() * 100) if not df_in.empty else None
    h.append('<tr>')
    for col_key, _ in IC_DETAIL_COLS:
        if col_key == "LOB_disp":
            h.append(f'<td style="{TD_S}background:{TOT_SEP};color:{TOT_FG};'
                     f'font-weight:bold;text-align:left;">Grand Total</td>')
        elif col_key == "IC Status":
            h.append(f'<td style="{TD_S}background:{MISS_BG};color:{MISS_FG};'
                     f'font-weight:bold;text-align:center;">Missed</td>')
        elif col_key == "SA_pct":
            val = f"{gt_sa:.1f}%" if gt_sa is not None else "&#8212;"
            bg_c, fg_c = color_sa_detail(gt_sa) if gt_sa is not None else (TOT_SEP, TOT_FG)
            h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                     f'font-weight:bold;text-align:right;">{val}</td>')
        elif col_key in IC_SUM_COLS and col_key in df_in.columns:
            total = pd.to_numeric(df_in[col_key], errors="coerce").sum()
            h.append(f'<td style="{TD_S}background:{TOT_SEP};color:{TOT_FG};'
                     f'font-weight:bold;text-align:right;">{total:.1f}</td>')
        else:
            h.append(f'<td style="{TD_S}background:{TOT_SEP};">&nbsp;</td>')
    h.append('</tr></tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL
# ══════════════════════════════════════════════════════════════════════════════
def build_banner(em=False):
    bi  = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs  = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    inn = (f'<p style="{bi}">&#128202; Expedia VN — IC &amp; Staffing Attainment Report</p>'
           f'<p style="{bs}">As of: <strong>{report_date_s}</strong> &nbsp;|&nbsp; '
           f'Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp; '
           f'IC Target: &ge;{IC_TARGET:.0f}% &nbsp;|&nbsp; '
           f'SA Target: &ge;{SA_TARGET:.0f}%</p>')
    if em:
        return (f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
                f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
                f'{inn}</td></tr></table>')
    return (f'<div style="background:{BANNER_C};padding:10px 14px;'
            f'border-radius:4px;margin:0 0 14px;">{inn}</div>')

def build_all(em=False):
    parts = [build_banner(em)]
    d_range = f"{daily_from} → {daily_to}"

    # Section 1 — Daily pivot (no Total)
    parts.append(_sec("1", "Daily IC &amp; Staffing Attainment",
        f"IC % target &ge;{IC_TARGET:.0f}%. IC Fail: red if &gt;{IC_FAIL_RED}, green otherwise. "
        f"Range: {d_range}.", em))
    parts.append(render_pivot_table(
        daily_piv, daily_periods, is_daily=True, show_total=False, em=em))
    parts.append(lgd(inline=em))
    parts.append(spacer(em))

    # Section 2 — Excalibur failed intervals
    parts.append(_sec("2", "Daily — Failed Intervals (Excalibur)",
        f"IC=0 from excalibur_raw. Forecast Productive = Total PSP. "
        f"SA: &ge;95% green | ~85.5% yellow | &lt;85.5% red. "
        f"Site = Productive Hrs &minus; MSP Site Wise: &gt;0 green | &lt;0 red | =0 yellow. "
        f"Range: {d_range}.", em))
    parts.append(render_excalibur_fail_table(fail_site, em=em))
    parts.append(spacer(em))

    # Section 3 — IC HCM Details
    parts.append(_sec("3", "Daily — Failed Intervals Detail (IC HCM Log)",
        f"IC=Missed from IC_HCM_Details_Log. "
        f"Req Heads = Site Req Heads (HCM). "
        f"Actual Heads: red if &lt; Scheduled OpenTime. "
        f"Actual Break/Lunch/T&amp;C: red if &ge; Scheduled. "
        f"Leave/NCNS: red if &gt; 0. "
        f"Range: {d_range}.", em))
    parts.append(render_ic_details_table(df_ic_fail, em=em))
    parts.append(spacer(em))

    # Section 4 — Weekly
    parts.append(_sec("4", f"Weekly IC &amp; Staffing Attainment — Last {N_WEEKLY_WEEKS} Weeks",
        f"IC Pass/Fail + IC % + SA % by week up to {daily_to}.", em))
    parts.append(render_pivot_table(
        weekly_piv, weekly_periods, is_daily=False, show_total=False, em=em))  # ← False
    parts.append(lgd(inline=em))
    parts.append(spacer(em))

    # Section 5 — Monthly
    parts.append(_sec("5", f"Monthly IC &amp; Staffing Attainment — Last {N_MONTHLY_MONTHS} Months",
        f"IC Pass/Fail + IC % + SA % by month up to {daily_to}.", em))
    parts.append(render_pivot_table(
        monthly_piv, monthly_periods, is_daily=False, show_total=False, em=em))  # ← False
    parts.append(lgd(inline=em))

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# EMAIL GREETING / SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear Team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find the Expedia VN IC &amp; Staffing Attainment Report
    as of <strong>{report_date_s}</strong> ({daily_from} → {daily_to}).
</p>
<ul style="{FONT}font-size:12px;margin:0 0 16px;padding-left:20px;line-height:1.8">
    <li><strong>IC %</strong> — Pass / Total Intervals. Target &ge;{IC_TARGET:.0f}%</li>
    <li><strong>IC Fail</strong> — Red if &gt;{IC_FAIL_RED}, Green if &le;{IC_FAIL_RED} (daily)</li>
    <li><strong>SA %</strong> — Total Productive Hrs / Total PSP</li>
    <li><strong>Sec 2 Site cols</strong> — Productive Hrs &minus; MSP Site Wise: &gt;0 green | &lt;0 red | =0 yellow</li>
    <li><strong>Sec 3 coloring</strong> — Actual Heads &lt; Scheduled OpenTime → red |
        Actual Break/Lunch/T&amp;C &ge; Scheduled → red |
        Leave/NCNS &gt; 0 → red</li>
    <li><span style="background:{MET_BG};color:{MET_FG};padding:1px 6px;font-weight:bold">Green</span> Met &nbsp;
        <span style="background:{WARN_BG};color:{WARN_FG};padding:1px 6px;font-weight:bold">Yellow</span> Near &nbsp;
        <span style="background:{MISS_BG};color:{MISS_FG};padding:1px 6px;font-weight:bold">Red</span> Miss</li>
</ul>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 12px;">
"""

def signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;font-weight:bold;margin:0 0 2px;">BI Associate</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:8px;">
    Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: excalibur_raw.parquet + IC_HCM_Details_Log.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb  = ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
           f"<style>{CSS}</style></head><body>"
           + build_all(em=False) + "</body></html>")
    esc = nb.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{esc}" style="width:100%;border:none;min-height:900px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'))
    print("✓ Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    html_body = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + greeting() + build_all(em=True) + signature() + "</div>"
    )

    def send_auto(to, cc, subject, html_body, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe"
                     for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("⏳ Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol   = win32com.client.Dispatch("Outlook.Application")
            ns   = ol.GetNamespace("MAPI"); ns.Logon()
            mail = ol.CreateItem(0)
            mail.To       = to
            mail.CC       = cc
            mail.Subject  = subject
            mail.HTMLBody = html_body
            mail.Send()
            print(f"✓ Sent → {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("✓ Outlook closed")
                except: pass

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, html_body, quit_after=True)

✓ AUTO D-1 = 2026-06-02
📂 Loading excalibur_raw.parquet...
✓ excalibur: 54,096 | 2025-12-29 → 2026-06-07
📂 Loading IC_HCM_Details_Log.parquet...
✓ IC Details: 12,321 | 2025-12-01 → 2026-06-02
  Excalibur max PST Date: 2026-06-07 → daily_to: 2026-06-06
✓ Daily range: 2026-06-01 → 2026-06-06
✓ Subject    : Expedia VN — IC & Staffing Attainment Report as of 02-Jun-2026
⏳ Building Daily pivot...
✓ Daily: ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05', '2026-06-06']
⏳ Building Excalibur failed intervals...
✓ Excalibur failed: 30
⏳ Building IC Details failed intervals...
✓ IC Details failed: 28
⏳ Building Weekly pivot...
✓ Weekly (8): ['04/13~04/19', '04/20~04/26', '04/27~05/03', '05/04~05/10', '05/11~05/17', '05/18~05/24', '05/25~05/31', '06/01~06/07']
⏳ Building Monthly pivot...
✓ Monthly (6): ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


✓ Display done
